In [1]:
# import os

# for root, dirs, files in os.walk("/kaggle/input/datasets/"):
#     for file in files:
#         print(os.path.join(root, file))

In [2]:
import pickle
import numpy as np

BASE = "/kaggle/input/datasets/ayushaabbas/patchcore-outputs/kaggle/working"

# Load full results
with open(f"{BASE}/patchcore_full_results.pkl", "rb") as f:
    results = pickle.load(f)

# Load memory banks
memory_banks = {}
for cat in results.keys():
    memory_banks[cat] = np.load(f"{BASE}/membank_{cat}.npy")

print("✓ Categories:", list(results.keys()))
print("✓ Memory banks:", {k: v.shape for k, v in memory_banks.items()})
print("✓ Sample path:", results['bottle']['image_paths'][:2])

✓ Categories: ['bottle', 'cable', 'capsule', 'carpet', 'grid', 'hazelnut', 'leather', 'metal_nut', 'pill', 'screw', 'tile', 'toothbrush', 'transistor', 'wood', 'zipper']
✓ Memory banks: {'bottle': (1638, 1536), 'cable': (1756, 1536), 'capsule': (1716, 1536), 'carpet': (2195, 1536), 'grid': (2069, 1536), 'hazelnut': (3065, 1536), 'leather': (1920, 1536), 'metal_nut': (1724, 1536), 'pill': (2093, 1536), 'screw': (2508, 1536), 'tile': (1803, 1536), 'toothbrush': (470, 1536), 'transistor': (1669, 1536), 'wood': (1936, 1536), 'zipper': (1881, 1536)}
✓ Sample path: ['/kaggle/input/datasets/ipythonx/mvtec-ad/bottle/test/broken_large/000.png'
 '/kaggle/input/datasets/ipythonx/mvtec-ad/bottle/test/broken_large/001.png']


In [3]:
from PIL import Image
import torch
import torch.nn as nn 
import timm
import numpy as np
from torchvision import transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Backbone
backbone = timm.create_model('wide_resnet50_2', pretrained=True, features_only=True)
backbone = backbone.to(device)
backbone.eval()

# Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

print("✓ Imports and backbone ready")

model.safetensors:   0%|          | 0.00/276M [00:00<?, ?B/s]

✓ Imports and backbone ready


In [4]:

def extract_features(image_path):
    """Extract patch features from a single image"""
    img = Image.open(image_path).convert("RGB")
    img_tensor = transform(img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        features = backbone(img_tensor)
    
    # Use layer 2 and 3 (indices 2 and 3) - mid level features
    f2 = features[2]  # shape: [1, 512, 28, 28]
    f3 = features[3]  # shape: [1, 1024, 14, 14]
    
    # Upsample f3 to match f2 spatial size
    f3_up = nn.functional.interpolate(f3, size=f2.shape[-2:], mode='bilinear', align_corners=False)
    
    # Concatenate along channel dimension
    combined = torch.cat([f2, f3_up], dim=1)  # shape: [1, 1536, 28, 28]
    
    # Reshape to patch vectors: [num_patches, feature_dim]
    b, c, h, w = combined.shape
    patches = combined.permute(0, 2, 3, 1).reshape(-1, c)  # [784, 1536]
    
    return patches.cpu().numpy()

In [5]:
def score_image(image_path, nn_index, k=9):
    """Compute anomaly score and heatmap for a test image"""
    patches = extract_features(image_path)
    
    # Find distance to k nearest normal patches in memory bank
    distances, _ = nn_index.kneighbors(patches)
    
    # Average over k neighbours
    anomaly_map = distances.mean(axis=1).reshape(28, 28)
    
    # Image level score = max patch distance
    image_score = anomaly_map.max()
    
    return image_score, anomaly_map


In [6]:
import pickle
import numpy as np

BASE = "/kaggle/input/datasets/ayushaabbas/patchcore-outputs/kaggle/working"

with open(f"{BASE}/patchcore_full_results.pkl", "rb") as f:
    results = pickle.load(f)

memory_banks = {}
for cat in results.keys():
    memory_banks[cat] = np.load(f"{BASE}/membank_{cat}.npy")

print("✓ Results loaded:", list(results.keys()))
print("✓ Memory banks loaded")

✓ Results loaded: ['bottle', 'cable', 'capsule', 'carpet', 'grid', 'hazelnut', 'leather', 'metal_nut', 'pill', 'screw', 'tile', 'toothbrush', 'transistor', 'wood', 'zipper']
✓ Memory banks loaded


In [7]:
"""
Human-in-the-Loop Feedback Loop — Complete Save Version
=========================================================
Saves every parameter that could be needed for paper writing,
plots, and analysis. Run once, never again.

Requirements (must be run in earlier cells first):
  - from PIL import Image
  - import torch, torch.nn as nn, timm
  - from torchvision import transforms
  - backbone, transform, device defined
  - extract_features() and score_image() defined
  - results and memory_banks loaded from patchcore-outputs dataset
"""

import numpy as np
import pickle
import os
import time
from copy import deepcopy
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (roc_auc_score, precision_recall_fscore_support,
                              confusion_matrix, roc_curve,
                              average_precision_score)

# ── CONFIG ────────────────────────────────────────────────────
EXPERIMENT_CATS = ['grid']
N_ROUNDS        = 30
SEED            = 42
SAVE_DIR        = "/kaggle/working"


# ── HELPERS ───────────────────────────────────────────────────

def build_nn(memory_bank, k=9):
    nn_idx = NearestNeighbors(n_neighbors=k, metric='euclidean',
                              algorithm='ball_tree', n_jobs=-1)
    nn_idx.fit(memory_bank)
    return nn_idx


def score_all(image_paths, nn_index):
    scores = []
    for path in image_paths:
        s, _ = score_image(str(path), nn_index)
        scores.append(s)
    return np.array(scores)


def compute_metrics_at_threshold(labels, scores, threshold):
    preds = (scores >= threshold).astype(int)
    prec, rec, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary', zero_division=0)
    cm = confusion_matrix(labels, preds)
    tn, fp, fn, tp = cm.ravel()
    return {
        'threshold': float(threshold),
        'precision': float(prec),
        'recall':    float(rec),
        'f1':        float(f1),
        'tp': int(tp), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn),
        'accuracy':  float((tp + tn) / (tp + tn + fp + fn))
    }


def apply_fp_correction(img_path, memory_bank, nn_index, max_bank_size=2500):
    new_patches = extract_features(str(img_path))
    memory_bank = np.concatenate([memory_bank, new_patches], axis=0)
    
    if len(memory_bank) > max_bank_size:
        indices = np.random.choice(len(memory_bank), max_bank_size, replace=False)
        memory_bank = memory_bank[indices]
    
    nn_index = build_nn(memory_bank)
    return memory_bank, nn_index, len(new_patches)


def apply_fn_correction(img_path, memory_bank, nn_index, k_remove=5):
    patches   = extract_features(str(img_path))
    dists, _  = nn_index.kneighbors(patches, n_neighbors=1)
    anchor    = patches[np.argmax(dists.squeeze())].reshape(1, -1)
    k_actual  = min(k_remove, len(memory_bank) - 10)
    _, rm_idx = nn_index.kneighbors(anchor, n_neighbors=k_actual)
    keep_mask = np.ones(len(memory_bank), dtype=bool)
    keep_mask[rm_idx.flatten()] = False
    n_removed   = int((~keep_mask).sum())
    memory_bank = memory_bank[keep_mask]
    nn_index    = build_nn(memory_bank)
    return memory_bank, nn_index, n_removed


def compute_aucc(auroc_curve, baseline_auroc):
    gains = [max(0.0, a - baseline_auroc) for a in auroc_curve[1:]]
    if not gains or max(gains) == 0:
        return 0.0
    max_possible = len(gains) * (1.0 - baseline_auroc)
    return float(np.trapezoid(gains) / max_possible) if max_possible > 0 else 0.0


# ── MAIN FEEDBACK LOOP ────────────────────────────────────────

def run_feedback_loop(category, strategy, n_rounds=N_ROUNDS, seed=SEED):
    """
    Returns a comprehensive dict with everything needed for paper writing.
    """
    rng         = np.random.default_rng(seed)
    start_time  = time.time()

    r           = results[category]
    image_paths = r['image_paths']
    labels      = r['labels']
    subtypes    = r['subtypes']
    memory_bank = deepcopy(memory_banks[category])
    nn_index    = build_nn(memory_bank)

    # ── Baseline ────────────────────────────────────────────
    scores         = score_all(image_paths, nn_index)
    baseline_auroc = roc_auc_score(labels, scores)
    threshold      = np.percentile(scores[labels == 0], 90)
    baseline_ap    = average_precision_score(labels, scores)
    baseline_metrics = compute_metrics_at_threshold(labels, scores, threshold)
    fpr, tpr, _    = roc_curve(labels, scores)

    print(f"    Baseline AUROC: {baseline_auroc:.4f}  AP: {baseline_ap:.4f}")

    # ── Per-round storage ────────────────────────────────────
    auroc_curve         = [baseline_auroc]
    ap_curve            = [baseline_ap]
    f1_curve            = [baseline_metrics['f1']]
    precision_curve     = [baseline_metrics['precision']]
    recall_curve        = [baseline_metrics['recall']]
    threshold_curve     = [float(threshold)]
    n_corrections       = [0]
    n_fp_corrections    = [0]
    n_fn_corrections    = [0]
    bank_size_curve     = [len(memory_bank)]
    scores_per_round    = [scores.copy()]       # full score array each round
    correction_log      = []                    # detailed log per correction
    round_times         = []

    corrected   = set()
    total_fp    = 0
    total_fn    = 0

    for round_idx in range(n_rounds):
        t0 = time.time()

        scores    = score_all(image_paths, nn_index)
        threshold = np.percentile(scores[labels == 0], 90)

        fp_pool = [(scores[i], image_paths[i], 'FP', subtypes[i])
                   for i in range(len(image_paths))
                   if labels[i] == 0 and scores[i] >= threshold
                   and str(image_paths[i]) not in corrected]

        fn_pool = [(scores[i], image_paths[i], 'FN', subtypes[i])
                   for i in range(len(image_paths))
                   if labels[i] == 1 and scores[i] < threshold
                   and str(image_paths[i]) not in corrected]

        pool = fp_pool + fn_pool
        if not pool:
            print(f"    Round {round_idx+1:3d}: no errors remain — stopping early")
            final_val = auroc_curve[-1]
            while len(auroc_curve) < n_rounds + 1:
                auroc_curve.append(final_val)
                ap_curve.append(ap_curve[-1])
                f1_curve.append(f1_curve[-1])
                precision_curve.append(precision_curve[-1])
                recall_curve.append(recall_curve[-1])
                threshold_curve.append(threshold_curve[-1])
                n_corrections.append(n_corrections[-1])
                n_fp_corrections.append(n_fp_corrections[-1])
                n_fn_corrections.append(n_fn_corrections[-1])
                bank_size_curve.append(bank_size_curve[-1])
                scores_per_round.append(scores_per_round[-1])
                round_times.append(0.0)
            break

        # Select correction target
        if strategy == 'passive':
            idx = rng.integers(len(pool))
            score_val, img_path, ctype, subtype = pool[idx]
        else:
            pool_sorted = sorted(pool, key=lambda x: abs(x[0] - threshold))
            score_val, img_path, ctype, subtype = pool_sorted[0]

        # Apply correction
        if ctype == 'FP':
            memory_bank, nn_index, n_patches = apply_fp_correction(
                img_path, memory_bank, nn_index)
            total_fp += 1
        else:
            memory_bank, nn_index, n_patches = apply_fn_correction(
                img_path, memory_bank, nn_index)
            total_fn += 1

        corrected.add(str(img_path))

        # Evaluate after correction
        scores      = score_all(image_paths, nn_index)
        curr_auroc  = roc_auc_score(labels, scores)
        curr_ap     = average_precision_score(labels, scores)
        new_thresh  = np.percentile(scores[labels == 0], 90)
        curr_met    = compute_metrics_at_threshold(labels, scores, new_thresh)
        round_time  = time.time() - t0

        # Log this correction
        correction_log.append({
            'round':       round_idx + 1,
            'type':        ctype,
            'image_path':  str(img_path),
            'subtype':     str(subtype),
            'score_before': float(score_val),
            'threshold':   float(threshold),
            'distance_to_threshold': float(abs(score_val - threshold)),
            'auroc_after': float(curr_auroc),
            'auroc_delta': float(curr_auroc - auroc_curve[-1]),
            'n_patches_changed': n_patches,
            'bank_size_after': len(memory_bank),
            'round_time_s': round_time,
        })

        auroc_curve.append(curr_auroc)
        ap_curve.append(curr_ap)
        f1_curve.append(curr_met['f1'])
        precision_curve.append(curr_met['precision'])
        recall_curve.append(curr_met['recall'])
        threshold_curve.append(float(new_thresh))
        n_corrections.append(len(corrected))
        n_fp_corrections.append(total_fp)
        n_fn_corrections.append(total_fn)
        bank_size_curve.append(len(memory_bank))
        scores_per_round.append(scores.copy())
        round_times.append(round_time)

        print(f"    Round {round_idx+1:3d} | {strategy:7s} | "
              f"{ctype} ({str(subtype):<20}) | "
              f"AUROC: {curr_auroc:.4f} | "
              f"corrections: {len(corrected):3d} | "
              f"bank: {len(memory_bank)}")

    total_time = time.time() - start_time
    aucc       = compute_aucc(auroc_curve, baseline_auroc)

    # Final ROC curve
    final_scores = scores_per_round[-1]
    fpr_final, tpr_final, _ = roc_curve(labels, final_scores)

    print(f"\n    ── Summary ──")
    print(f"    AUCC:        {aucc:.4f}")
    print(f"    Final AUROC: {auroc_curve[-1]:.4f}  (was {baseline_auroc:.4f})")
    print(f"    Improvement: {auroc_curve[-1] - baseline_auroc:+.4f}")
    print(f"    FP corrected: {total_fp}  FN corrected: {total_fn}")
    print(f"    Total time:   {total_time/60:.1f} min")

    return {
        # ── Core curves ──────────────────────────────────────
        'auroc_curve':       auroc_curve,        # AUROC after each round
        'ap_curve':          ap_curve,            # Average Precision each round
        'f1_curve':          f1_curve,            # F1 each round
        'precision_curve':   precision_curve,     # Precision each round
        'recall_curve':      recall_curve,        # Recall each round
        'threshold_curve':   threshold_curve,     # Decision threshold each round
        'n_corrections':     n_corrections,       # Cumulative corrections
        'n_fp_corrections':  n_fp_corrections,    # Cumulative FP corrections
        'n_fn_corrections':  n_fn_corrections,    # Cumulative FN corrections
        'bank_size_curve':   bank_size_curve,     # Memory bank size each round

        # ── Summary stats ────────────────────────────────────
        'baseline_auroc':    baseline_auroc,
        'baseline_ap':       baseline_ap,
        'baseline_metrics':  baseline_metrics,    # precision/recall/F1/TP/TN/FP/FN
        'final_auroc':       auroc_curve[-1],
        'final_ap':          ap_curve[-1],
        'improvement':       auroc_curve[-1] - baseline_auroc,
        'aucc':              aucc,

        # ── ROC curves ───────────────────────────────────────
        'roc_fpr_baseline':  fpr,                 # ROC before feedback
        'roc_tpr_baseline':  tpr,
        'roc_fpr_final':     fpr_final,           # ROC after feedback
        'roc_tpr_final':     tpr_final,

        # ── Per-round raw scores ─────────────────────────────
        'scores_per_round':  scores_per_round,    # Full score array each round

        # ── Correction log ───────────────────────────────────
        'correction_log':    correction_log,      # Detailed log per correction

        # ── Metadata ─────────────────────────────────────────
        'category':          category,
        'strategy':          strategy,
        'n_rounds_run':      len(auroc_curve) - 1,
        'total_fp_corrected': total_fp,
        'total_fn_corrected': total_fn,
        'total_time_s':      total_time,
        'round_times_s':     round_times,
        'image_paths':       image_paths,         # needed for visualisation
        'labels':            labels,
        'subtypes':          subtypes,
    }


# ── MAIN EXPERIMENT LOOP ──────────────────────────────────────

# Auto-resume: load any already-completed categories
feedback_results = {}
for cat in EXPERIMENT_CATS:
    path = os.path.join(SAVE_DIR, f"hitl_{cat}.pkl")
    if os.path.exists(path):
        with open(path, "rb") as f:
            feedback_results[cat] = pickle.load(f)
        print(f"✓ Loaded existing: {cat}")

categories_to_run = [c for c in EXPERIMENT_CATS if c not in feedback_results]
print(f"\nAlready done: {list(feedback_results.keys())}")
print(f"Still to run: {categories_to_run}\n")

for category in categories_to_run:
    print(f"\n{'='*60}")
    print(f"  {category.upper()}")
    print(f"{'='*60}")
    feedback_results[category] = {}

    for strategy in ['passive', 'active']:
        print(f"\n  Strategy: {strategy.upper()}")
        feedback_results[category][strategy] = run_feedback_loop(
            category, strategy, n_rounds=N_ROUNDS, seed=SEED
        )

    # Save immediately after both strategies complete for this category
    cat_path = os.path.join(SAVE_DIR, f"hitl_{category}.pkl")
    with open(cat_path, "wb") as f:
        pickle.dump(feedback_results[category], f)
    print(f"\n  ✓ Saved hitl_{category}.pkl")

# Final merge
final_path = os.path.join(SAVE_DIR, "hitl_feedback_results.pkl")
with open(final_path, "wb") as f:
    pickle.dump(feedback_results, f)
print(f"\n✓ All saved to {final_path}")


# ── SUMMARY TABLE ─────────────────────────────────────────────

print(f"\n{'='*80}")
print(f"{'Category':<15} {'Strategy':<10} {'Baseline':>9} {'Final':>7} "
      f"{'Improve':>9} {'AUCC':>7} {'FP':>5} {'FN':>5}")
print(f"{'='*80}")

for cat in EXPERIMENT_CATS:
    if cat not in feedback_results:
        continue
    for strategy in ['passive', 'active']:
        r = feedback_results[cat][strategy]
        print(f"{cat:<15} {strategy:<10} {r['baseline_auroc']:>9.4f} "
              f"{r['final_auroc']:>7.4f} {r['improvement']:>+9.4f} "
              f"{r['aucc']:>7.4f} "
              f"{r['total_fp_corrected']:>5} {r['total_fn_corrected']:>5}")
    print(f"{'':-<80}")


Already done: []
Still to run: ['grid']


  GRID

  Strategy: PASSIVE
    Baseline AUROC: 0.8839  AP: 0.9543
    Round   1 | passive | FP (good                ) | AUROC: 0.8521 | corrections:   1 | bank: 2500
    Round   2 | passive | FN (thread              ) | AUROC: 0.8505 | corrections:   2 | bank: 2495
    Round   3 | passive | FN (metal_contamination ) | AUROC: 0.8521 | corrections:   3 | bank: 2490
    Round   4 | passive | FN (glue                ) | AUROC: 0.8530 | corrections:   4 | bank: 2485
    Round   5 | passive | FN (glue                ) | AUROC: 0.8638 | corrections:   5 | bank: 2480
    Round   6 | passive | FN (thread              ) | AUROC: 0.8713 | corrections:   6 | bank: 2475
    Round   7 | passive | FP (good                ) | AUROC: 0.8655 | corrections:   7 | bank: 2500
    Round   8 | passive | FN (metal_contamination ) | AUROC: 0.8655 | corrections:   8 | bank: 2495
    Round   9 | passive | FN (bent                ) | AUROC: 0.8680 | corrections:   9 | b